<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/2D_Smiles_to_3D_SDF_success.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 42.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Phase5_AI - FDA_20_Metadata.csv to Phase5_AI - FDA_20_Metadata (1).csv


In [3]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import os

# 1. Load your CSV file
# Make sure the filename matches exactly what you uploaded
csv_filename = 'Phase5_AI - FDA_20_Metadata.csv'
df = pd.read_csv(csv_filename)

# 2. CHANGE THIS to match your CSV's exact structure column name (e.g., 'SMILES', 'Structure', 'smiles')
STRUCTURE_COLUMN = 'SMILES'
NAME_COLUMN = df.columns[0] # Defaults to the first column for naming the molecules, change if needed

output_sdf = 'FDA_20_prepared_3d.sdf'
writer = Chem.SDWriter(output_sdf)

success_count = 0

print("Starting 3D conformation generation...")

for index, row in df.iterrows():
    smiles = str(row[STRUCTURE_COLUMN]).strip()
    mol_name = str(row[NAME_COLUMN]).strip()

    # Parse SMILES to RDKit Molecule object
    mol = Chem.MolFromSmiles(smiles)

    if mol is not None:
        # Set molecule name property for the SDF file
        mol.SetProp("_Name", mol_name)

        # 1. Add implicit hydrogens (crucial for accurate 3D geometry)
        mol = Chem.AddHs(mol)

        # 2. Generate initial 3D coordinates using ETKDG method
        # ETKDG is highly reliable for small-molecule drugs
        conformer_id = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())

        if conformer_id >= 0:
            # 3. Minimize energy using MMFF94 force field for clean 3D structures
            try:
                AllChem.MMFFOptimizeMolecule(mol)
                writer.write(mol)
                success_count += 1
                print(index + 1, f"Successfully processed: {mol_name}")
            except Exception as e:
                print(index + 1, f"Failed optimization for {mol_name}: {e}")
        else:
            print(index + 1, f"Failed 3D embedding for {mol_name}")
    else:
        print(index + 1, f"Invalid SMILES string for {mol_name}")

writer.close()
print(f"\n Done! Successfully saved {success_count}/20 compounds to '{output_sdf}'.")

Starting 3D conformation generation...
1 Successfully processed: AI_INV_Vidarabine_1
2 Successfully processed: AI_INV_Vidarabine_2
3 Successfully processed: AI_INV_Vidarabine_3
4 Successfully processed: AI_INV_Vidarabine_4
5 Successfully processed: AI_INV_Vidarabine_5
6 Successfully processed: AI_INV_Vidarabine_6
7 Successfully processed: AI_INV_Vidarabine_7
8 Successfully processed: AI_INV_Vidarabine_8
9 Successfully processed: AI_INV_Vidarabine_9
10 Successfully processed: AI_INV_Vidarabine_10
11 Successfully processed: AI_INV_Remdesivir_1
12 Successfully processed: AI_INV_Remdesivir_2
13 Successfully processed: AI_INV_Remdesivir_3
14 Successfully processed: AI_INV_Remdesivir_4
15 Successfully processed: AI_INV_Remdesivir_5
16 Successfully processed: AI_INV_Remdesivir_6
17 Successfully processed: AI_INV_Remdesivir_7
18 Successfully processed: AI_INV_Remdesivir_8
19 Successfully processed: AI_INV_Remdesivir_9
20 Successfully processed: AI_INV_Remdesivir_10

 Done! Successfully saved 2